# Neural Networks Report

In [ ]:
import sys
import os


# for google colab: 
if 'google.colab' in sys.modules or os.path.exists('/content'):
    if not os.path.exists('/content/NEURALENJOYERS'):
        token=""
        os.system(f'git clone https://{token}@github.com/Projekty-IiE/NeuralEnjoyers.git /content/NEURALENJOYERS')
    else:
        os.system('git -C /content/NEURALENJOYERS pull')
    sys.path.insert(0, '/content/NEURALENJOYERS/Klasyfikacyjny')
    os.chdir('/content/NEURALENJOYERS')

# external
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import csv

# local
import mnist_loader
import network
import network2

# 1. Wczytanie danych
training_data, validation_data, test_data = mnist_loader.load_data_wrapper()
    
# Rzutowanie na listy dla Pythona 3
training_data = list(training_data)
validation_data = list(validation_data)
test_data = list(test_data)

# base parameters
base_sizes = [784, 30, 10]
base_epochs = 10
base_batch = 10
base_eta = 0.5
base_lmbda = 5.0


In [ ]:
def run_experiment(param_name, values_to_test, param_type='eta'):
    results = []
    runs_per_config = 3 # Repeat 3 times for non-deterministic averaging
    
    # Save to CSV as we go 
    with open('Klasyfikacyjny/backup_results.csv', mode='a', newline='') as file:
        writer = csv.writer(file)
        
        for val in values_to_test:
            print(f"\n>> Testing {param_name}: {val}")
            for run in range(runs_per_config):
                print(f"   Run {run+1}/{runs_per_config}...")
                
                # 1. Reset all parameters to our "Base" Control Variables
                current_eta = base_eta
                current_batch = base_batch
                current_lmbda = base_lmbda
                current_epochs = base_epochs
                current_sizes = base_sizes
                current_train = training_data
                current_test = test_data
                
                # 2. OVERRIDE the specific parameter we are testing
                if param_type == 'eta': current_eta = val
                elif param_type == 'batch': current_batch = val
                elif param_type == 'lmbda': current_lmbda = val
                elif param_type == 'epochs': current_epochs = val
                elif param_type == 'neurons': current_sizes = [784, val, 10]
                elif param_type == 'layers': current_sizes = val # Pass a whole list like [784, 30, 30, 10]
                elif param_type == 'train_size': current_train = training_data[:val] # Slice the list
                elif param_type == 'test_size': current_test = test_data[:val] # Slice the list
                
                # 3. Initialize fresh network with (potentially new) size
                net = network2.Network(current_sizes, cost=network2.CrossEntropyCost)
                
                # 4. Train
                eval_cost, eval_acc, train_cost, train_acc = net.SGD(
                    training_data=current_train, epochs=current_epochs, 
                    mini_batch_size=current_batch, eta=current_eta, lmbda=current_lmbda, 
                    evaluation_data=current_test, monitor_evaluation_accuracy=True,
                    monitor_training_accuracy=True
                )
                
                # 5. Calculate percentage accuracy
                final_train = (train_acc[-1] / len(current_train)) * 100
                final_test = (eval_acc[-1] / len(current_test)) * 100
                
                # 6. Save results
                # We convert 'val' to a string just in case it's a list (like for layers) so it saves cleanly
                results.append({'Value': str(val), 'Run': run+1, 'Train_Acc': final_train, 'Test_Acc': final_test})
                writer.writerow([param_name, str(val), run+1, final_train, final_test])
                file.flush()
                
    # Return as a pandas dataframe for plotting
    return pd.DataFrame(results)

## Impact of changing parameters

### Learning Rate 

In [ ]:
# 1. Run the test
df_eta = run_experiment('Learning_Rate', [0.1, 0.5, 3.0, 10.0], param_type='eta')

# 2. Calculate averages across the 3 runs
avg_results = df_eta.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results['Value'], avg_results['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results['Value'], avg_results['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Learning Rate (eta) Impact')
plt.xlabel('eta')
plt.ylabel('Effectiveness (%)')
plt.legend()
plt.grid(True)
plt.show()

### Mini-batch Size

In [ ]:
# 1. Run the test
df_batch = run_experiment('Batch_Size', [1, 10, 50, 100], param_type='batch')

# 2. Calculate averages across the 3 runs
avg_results_batch = df_batch.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results_batch['Value'], avg_results_batch['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_batch['Value'], avg_results_batch['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ rozmiaru mini-paczki (Batch Size) na skuteczność')
plt.xlabel('Rozmiar mini-paczki')
plt.ylabel('Skuteczność (%)')
plt.legend()
plt.grid(True)
plt.show()

### L2 Regularization 

In [ ]:
# 1. Run the test
df_lmbda = run_experiment('L2_Regularization', [0.0, 1.0, 5.0, 10.0], param_type='lmbda')

# 2. Calculate averages across the 3 runs
avg_results_lmbda = df_lmbda.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results_lmbda['Value'], avg_results_lmbda['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_lmbda['Value'], avg_results_lmbda['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ parametru regularyzacji L2 (lambda) na skuteczność')
plt.xlabel('Parametr Lambda')
plt.ylabel('Skuteczność (%)')
plt.legend()
plt.grid(True)
plt.show()

### Number of Epochs

In [ ]:
# 1. Run the test
df_epochs = run_experiment('Epochs', [5, 10, 20, 30], param_type='epochs')

# 2. Calculate averages across the 3 runs
avg_results_epochs = df_epochs.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results_epochs['Value'], avg_results_epochs['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_epochs['Value'], avg_results_epochs['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ liczby epok na skuteczność')
plt.xlabel('Liczba epok')
plt.ylabel('Skuteczność (%)')
plt.legend()
plt.grid(True)
plt.show()

### Number of Hidden Neurons

In [ ]:
# 1. Run the test
df_neurons = run_experiment('Hidden_Neurons', [10, 30, 60, 100], param_type='neurons')

# 2. Calculate averages across the 3 runs
avg_results_neurons = df_neurons.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results_neurons['Value'], avg_results_neurons['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_neurons['Value'], avg_results_neurons['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ liczby neuronów w warstwie ukrytej na skuteczność')
plt.xlabel('Liczba neuronów')
plt.ylabel('Skuteczność (%)')
plt.legend()
plt.grid(True)
plt.show()

### Number of Hidden Layers

In [ ]:
# 1. Run the test
layer_configs = [[784, 30, 10], [784, 30, 30, 10], [784, 30, 30, 30, 10], [784, 15, 15, 15, 15, 10]]
df_layers = run_experiment('Number_of_Layers', layer_configs, param_type='layers')

# 2. Calculate averages across the 3 runs
avg_results_layers = df_layers.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(10, 5)) # Made slightly wider to fit the text labels
plt.plot(avg_results_layers['Value'], avg_results_layers['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_layers['Value'], avg_results_layers['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ liczby warstw ukrytych na skuteczność')
plt.xlabel('Architektura sieci')
plt.ylabel('Skuteczność (%)')
plt.xticks(rotation=15) # Tilts the text slightly so the big lists don't overlap
plt.legend()
plt.grid(True)
plt.show()

### Training Data Size

In [ ]:
# 1. Run the test
df_train_size = run_experiment('Training_Data_Size', [1000, 10000, 25000, 50000], param_type='train_size')

# 2. Calculate averages across the 3 runs
avg_results_train_size = df_train_size.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results_train_size['Value'], avg_results_train_size['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_train_size['Value'], avg_results_train_size['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ wielkości próby uczącej na skuteczność')
plt.xlabel('Rozmiar próby uczącej (ilość obrazków)')
plt.ylabel('Skuteczność (%)')
plt.legend()
plt.grid(True)
plt.show()

### Test Data Size

In [ ]:
# 1. Run the test
df_test_size = run_experiment('Test_Data_Size', [1000, 3000, 5000, 10000], param_type='test_size')

# 2. Calculate averages across the 3 runs
avg_results_test_size = df_test_size.groupby('Value')[['Train_Acc', 'Test_Acc']].mean().reset_index()

# 3. Draw the graph
plt.figure(figsize=(8, 5))
plt.plot(avg_results_test_size['Value'], avg_results_test_size['Train_Acc'], marker='o', label='Próba Ucząca (Train)')
plt.plot(avg_results_test_size['Value'], avg_results_test_size['Test_Acc'], marker='s', label='Próba Testowa (Test)')
plt.title('Wpływ wielkości próby testowej na skuteczność')
plt.xlabel('Rozmiar próby testowej')
plt.ylabel('Skuteczność (%)')
plt.legend()
plt.grid(True)
plt.show()